In [ ]:
import os
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GCNConv, global_mean_pool

from rdkit import Chem

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
random_state=25


random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Imposto PyTorch in modalità deterministica
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # per determinismo cuBLAS CUDA 11+
torch.use_deterministic_algorithms(True)

In [ ]:
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db=client["PolymerPrediction"]
collection=db["biceranoTg"]
target="Tg"
#print(collection.count_documents({}))

docs = list(collection.find({}))
df = pd.DataFrame(docs)

# espando il dizionario "properties"
prop_df = pd.json_normalize(df["properties"])

# integro nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

# Creo una versione monomerica, saturata con H (baseline OGB), in un'altra colonna per mantenere lo SMILES originale
df["smiles_monomer"] = df["smiles"].str.replace("[*]", "", regex=False)  # es: "CCO"

# rimuovo righe senza SMILES o target o con SMILES malformato
df = df.dropna(subset=["smiles", target])
df_valid = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
df_valid.describe()

## Featurizzazione dei polimeri come grafi 

Qui uso la funzione `smiles2graph` della libreria OGB (Open Graph Benchmark) per
convertire i SMILES dei polimeri in grafi utilizzabili da PyTorch Geometric.

In pratica:
- RDKit legge lo SMILES e costruisce la molecola
- OGB estrae feature per ogni atomo e ogni legame
- ottengo tensori (node_feat, edge_index, edge_feat) pronti per la GNN

In [ ]:
from ogb.utils.mol import smiles2graph
from torch_geometric.data import Data
import torch

def smiles_to_graph(smiles: str, y_value: float) -> Data:
    """
    Converte un SMILES in un grafo PyG usando la featurizzazione di OGB.

    Basato su: Open Graph Benchmark (Hu et al., NeurIPS 2020),
    tramite la funzione ogb.utils.mol.smiles2graph.
    """
    g = smiles2graph(smiles)

    # node_feat: array [num_nodes, num_node_features]
    x = torch.tensor(g["node_feat"], dtype=torch.float)

    # edge_index: array [2, num_edges]
    edge_index = torch.tensor(g["edge_index"], dtype=torch.long)

    # edge_attr: eventuali feature dei legami
    # edge_attr = torch.tensor(g["edge_feat"], dtype=torch.float)

    y = torch.tensor([y_value], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, y=y)
    return data

In [ ]:
def add_periodic_edge(data: Data, terminal_indices=None) -> Data:
    """
    Aggiunge un edge 'periodico' bidirezionale tra due nodi terminali.
    - se terminal_indices è None, collega 0 e N-1 (catena lineare).

    Tecnica usata nel paper 
    Representing Polymers as Periodic Graphs with Learned Descriptors for Accurate Polymer Property Predictions, 
    Antoniuk et al. 2022
    """
    x = data.x
    edge_index = data.edge_index
    num_nodes = x.size(0)

    if num_nodes < 2:
        return data

    if terminal_indices is None:
        i = 0
        j = num_nodes - 1
    else:
        i, j = terminal_indices

    periodic_edges = torch.tensor([[i, j],
                                   [j, i]], dtype=torch.long)  # shape [2, 2]
    edge_index = torch.cat([edge_index, periodic_edges], dim=1)
    data.edge_index = edge_index
    return data

In [ ]:
from rdkit import Chem

def build_trimer_smiles(smiles_monomer: str) -> str:
    #TODO
    return "placeholder"

In [ ]:
# 1) Monomero saturo (baseline)
def monomer_saturated_graph(smiles_monomer: str, y_value: float) -> Data:
    return smiles_to_graph(smiles_monomer, y_value)

# 2) Monomero periodico (test intermedio)
def monomer_periodic_graph(smiles_monomer: str, y_value: float) -> Data:
    data = smiles_to_graph(smiles_monomer, y_value)
    data = add_periodic_edge(data)
    return data

# 3) Trimer periodico (polymer graph stile Antoniuk)
def trimer_periodic_graph(smiles_monomer: str, y_value: float) -> Data:
    trimer_smiles = build_trimer_smiles(smiles_monomer)
    data = smiles_to_graph(trimer_smiles, y_value)
    data = add_periodic_edge(data)
    return data

In [ ]:
class PolymerDataset(InMemoryDataset):
    def __init__(self, dataframe, smiles_col, target_col,
                 mode="monomer",
                 root="pyg_polymer_dataset",
                 transform=None, pre_transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.smiles_col = smiles_col
        self.target_col = target_col
        self.mode = mode
        super().__init__(root=root, transform=transform, pre_transform=pre_transform)
        self.data, self.slices = self._process_dataframe()

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return []

    def download(self):
        pass

    def process(self):
        pass

    def _process_dataframe(self):
        data_list = []
        for idx, row in self.dataframe.iterrows():
            smiles = row[self.smiles_col]
            y_val = row[self.target_col]
            try:
                if self.mode == "monomer":
                    data = monomer_saturated_graph(smiles, y_val)
                elif self.mode == "monomer_periodic":
                    data = monomer_periodic_graph(smiles, y_val)
                elif self.mode == "trimer_periodic":
                    data = trimer_periodic_graph(smiles, y_val)
                else:
                    raise ValueError(f"Unknown mode: {self.mode}")
                data_list.append(data)
            except Exception as e:
                print(f"Errore su riga {idx} ({smiles}): {e}")
        data, slices = self.collate(data_list)
        return data, slices

In [ ]:
indices = np.arange(len(df_valid))

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=random_state)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=random_state)

df_train = df_valid.iloc[train_idx].reset_index(drop=True)
df_val   = df_valid.iloc[val_idx].reset_index(drop=True)
df_test  = df_valid.iloc[test_idx].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)


# Creazione dataset

In [ ]:
# 1) Monomero normale (baseline)
train_mono = PolymerDataset(
    df_train, "smiles_monomer", target, mode="monomer"
)
val_mono = PolymerDataset(
    df_val, "smiles_monomer", target, mode="monomer"
)
test_mono = PolymerDataset(
    df_test, "smiles_monomer", target, mode="monomer"
)

# 2) Monomero periodico
train_mono_per = PolymerDataset(
    df_train, "smiles_monomer", target, mode="monomer_periodic"
)
val_mono_per = PolymerDataset(
    df_val, "smiles_monomer", target, mode="monomer_periodic"
)
test_mono_per = PolymerDataset(
    df_test, "smiles_monomer", target, mode="monomer_periodic"
)

"""
# 3) Trimer periodico (polymer graph representation del paper) - per ora da aggiustare
train_tri_per = PolymerDataset(
    df_train, "smiles", target, mode="trimer_periodic"
)
val_tri_per = PolymerDataset(
    df_val, "smiles", target, mode="trimer_periodic"
)
test_tri_per = PolymerDataset(
    df_test, "smiles", target, mode="trimer_periodic"
)
"""

# Creazione Loader

In [ ]:
g = torch.Generator()
g.manual_seed(random_state)

def make_loaders(train_ds, val_ds, test_ds):
    return (
        DataLoader(train_ds, batch_size=32, shuffle=True, generator=g),
        DataLoader(val_ds,   batch_size=32, shuffle=False, generator=g),
        DataLoader(test_ds,  batch_size=32, shuffle=False, generator=g)
    )

train_loader_mono,     val_loader_mono,     test_loader_mono     = make_loaders(train_mono,     val_mono,     test_mono)
train_loader_mono_per, val_loader_mono_per, test_loader_mono_per = make_loaders(train_mono_per, val_mono_per, test_mono_per)
#train_loader_tri_per,  val_loader_tri_per,  test_loader_tri_per  = make_loaders(train_tri_per,  val_tri_per,  test_tri_per)


# imposto quale versione usare: monomero saturo, monomero periodico o trimero periodico
train_dataset=train_mono
train_loader = train_loader_mono
val_loader = val_loader_mono
test_loader = test_loader_mono

# Regressors

In [ ]:
class GINRegressor(nn.Module):
    def __init__(self, in_channels, h=64, layers=3):
        super().__init__()

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        nn_in = nn.Sequential(nn.Linear(in_channels, h), nn.ReLU(), nn.Linear(h, h))
        self.convs.append(GINConv(nn_in))
        self.bns.append(nn.LayerNorm(h))

        for _ in range(layers-1):
            nn_k = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
            self.convs.append(GINConv(nn_k))
            self.bns.append(nn.LayerNorm(h))

        self.lin1 = nn.Linear(h, h)
        self.lin2 = nn.Linear(h, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)

        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x).view(-1)

In [ ]:
class GCNRegressor(nn.Module):
    def __init__(self, in_channels, h=64, layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        # Primo layer GCN: da in_channels a h
        self.convs.append(GCNConv(in_channels, h))
        self.bns.append(nn.LayerNorm(h))

        # Altri layer GCN: da h a h
        for _ in range(layers - 1):
            self.convs.append(GCNConv(h, h))
            self.bns.append(nn.LayerNorm(h))

        self.lin1 = nn.Linear(h, h)
        self.lin2 = nn.Linear(h, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)

        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x).view(-1)

# Model and Optimizer Initialization

In [ ]:
in_channels = train_dataset[0].x.size(1)
model = GCNRegressor(in_channels=in_channels, h=64, layers=3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',      # voglio minimizzare la metrica di validazione
    factor=0.5,      # moltiplica il LR per 0.5 quando non migliora
    patience=10,     # dopo 10 epoche senza miglioramenti
)

# Training and Evaluation

In [ ]:
def train_epoch(model, loader):
    model.train()
    tot = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(out, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        tot += loss.item() * batch.num_graphs
    return tot / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    ys, preds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            ys.append(batch.y.cpu().numpy())
            preds.append(out.cpu().numpy())

    ys = np.concatenate(ys)
    preds = np.concatenate(preds)
    return mean_absolute_error(ys, preds), r2_score(ys, preds)

In [ ]:
best = None
best_mae = float("inf")
patience = 40
epochs_no_improve = 0

for epoch in range(1, 301):
    loss = train_epoch(model, train_loader)
    val_mae, val_r2 = evaluate(model, val_loader)

    scheduler.step(val_mae)

    if val_mae < best_mae - 1e-4:  # piccolo margine per evitare fluttuazioni numeriche
        best_mae = val_mae
        best = {
            "epoch": epoch,
            "state": model.state_dict(),
            "mae": val_mae,
            "r2": val_r2,
        }
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} "
              f"loss {loss:.3f} "
              f"val MAE {val_mae:.2f} "
              f"val R2 {val_r2:.3f}")

    # Early stopping
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
model.load_state_dict(best["state"])
test_mae, test_r2 = evaluate(model, test_loader)
print("TEST MAE:", test_mae)
print("TEST R2:", test_r2)
